> **Licence:** this notebook depends on Ultralytics (AGPL-3.0). The models it
> produces are covered by AGPL-3.0 for commercial closed-source use — read
> [`../LICENSES.md`](../LICENSES.md) before using the output in a product.

# Fire & Smoke Watch - train our own detector (free Kaggle T4)

Goal: produce `best_480.onnx`, the exact file `app/detector.py` loads. Training
here (not on the VM) because a 2-vCPU ARM box would need ~5-6 days per run.

**Before running**
1. Account -> Settings -> *Phone verification* (required for GPU).
2. Right panel -> *Accelerator*: **GPU T4 x1**, *Internet*: On.
3. Right panel -> *Add Input* -> search **`sayedgamal99/smoke-fire-detection-yolo`**
   (D-Fire: 21,527 images, CC0 licence, classes `smoke` and `fire`).
4. *Run All*. A 50-epoch run takes roughly 2-3 hours on a T4.

**Output**: `/kaggle/working/best.pt` and `/kaggle/working/best_480.onnx`.
Download them and drop the `.onnx` next to `app/` on the VM.

> Licence note: Ultralytics YOLO is AGPL-3.0. For a closed-source client product
> either buy their Enterprise licence or repeat this recipe with an Apache-2.0
> detector (YOLOX / NanoDet / RT-DETR) - the serving code only needs an ONNX
> `output0` shaped `[1, 4+nc, N]`.


In [ ]:
!pip -q install ultralytics==8.* onnx onnxruntime
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Locate the dataset and normalise it into the layout YOLO expects.
import glob, os, shutil, yaml
from pathlib import Path

root = Path("/kaggle/input")
yamls = sorted(glob.glob(str(root / "**" / "data.yaml"), recursive=True))
print("data.yaml files found:", yamls)

if yamls:
    cfg = yaml.safe_load(open(yamls[0]))
    base = Path(yamls[0]).parent
    # Kaggle mounts inputs read-only with absolute /kaggle/input paths baked in;
    # rewrite them so the paths point at the actual mount.
    def fix(p):
        p = str(p)
        for pre in ("../", "./"):
            if p.startswith(pre):
                p = p[len(pre):]
        return str((base / p).resolve() if not os.path.isabs(p) else p)
    data = {
        "path": str(base),
        "train": fix(cfg.get("train", "train/images")),
        "val": fix(cfg.get("val", cfg.get("valid", "valid/images"))),
        "test": fix(cfg.get("test", "test/images")),
        "names": cfg.get("names", {0: "smoke", 1: "fire"}),
    }
else:
    # Fallback: build the config from whatever split folders exist.
    imgs = [p for p in root.rglob("images") if p.is_dir()]
    print("image dirs:", [str(i) for i in imgs])
    def pick(*names):
        for n in names:
            for p in imgs:
                if n in str(p).lower():
                    return str(p)
        return None
    data = {"path": str(imgs[0].parent),
            "train": pick("train"), "val": pick("val", "valid"),
            "test": pick("test"), "names": {0: "smoke", 1: "fire"}}

# sanity: count images per split
for k in ("train", "val", "test"):
    v = data.get(k)
    if v and os.path.isdir(v):
        print(k, len(os.listdir(v)), "images ->", v)

data_path = Path("/kaggle/working/data.yaml")
yaml.safe_dump(data, open(data_path, "w"), sort_keys=False)
print(open(data_path).read())

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")          # small and CPU-friendly at serving time
results = model.train(
    data=str(data_path),
    epochs=50,
    imgsz=640,                      # train at 640, serve at 480
    batch=16,
    patience=12,
    optimizer="auto",
    seed=0,
    deterministic=True,
    plots=True,
    project="/kaggle/working/runs",
    name="fire_smoke",
)
print("best weights:", results.save_dir)

In [ ]:
from ultralytics import YOLO

best = "/kaggle/working/runs/fire_smoke/weights/best.pt"
m = YOLO(best)
metrics = m.val(data=str(data_path), imgsz=640, split="val")
print("mAP50    :", round(float(metrics.box.map50), 4))
print("mAP50-95 :", round(float(metrics.box.map), 4))
print("precision:", round(float(metrics.box.mp), 4))
print("recall   :", round(float(metrics.box.mr), 4))

In [ ]:
import shutil
from pathlib import Path
from ultralytics import YOLO

best = "/kaggle/working/runs/fire_smoke/weights/best.pt"
m = YOLO(best)
onnx_path = m.export(format="onnx", imgsz=480, half=False, dynamic=False, simplify=True)
shutil.copy(best, "/kaggle/working/best.pt")
shutil.copy(onnx_path, "/kaggle/working/best_480.onnx")

import onnxruntime as ort
s = ort.InferenceSession("/kaggle/working/best_480.onnx", providers=["CPUExecutionProvider"])
print("input :", s.get_inputs()[0].shape)
print("output:", s.get_outputs()[0].shape, "  # expect [1, 6, N] = 4 box + 2 classes")

# quick self-check on a couple of validation images with boxes drawn
import glob, cv2
from ultralytics import YOLO as Y
v = Y("/kaggle/working/best_480.onnx", task="detect")
files = sorted(glob.glob(str(Path(data["val"]) / "*.jpg")))[:4]
for i, f in enumerate(v.predict(files, imgsz=480, conf=0.35), start=1):
    out = f"/kaggle/working/check_{i}.jpg"
    cv2.imwrite(out, f.plot())
    print(out, f.boxes.cls.tolist(), [round(float(x), 3) for x in f.boxes.conf.tolist()])
print("DONE - download best_480.onnx and best.pt from the Output tab")

## After training

1. Download `best_480.onnx` (and keep `best.pt` for future re-exports).
2. On the VM: `scp best_480.onnx ubuntu@<vm>:~/fire-lab/` then restart the service.
3. Re-run the local proof, in this order:

```bash
./venv/bin/python -m pytest tests -q
./venv/bin/python tools/e2e_checks.py
./venv/bin/python tools/browser_e2e.py "http://127.0.0.1:8512/?k=$FIREWATCH_CODE"
```

If `[FAIL]` shows up anywhere, do not swap the weights into a client demo -
compare the new `results.csv` against the numbers recorded in `REPORT.md` first.
